# HYDRA L4 GPU Training & Validation

**Goal**: Train HYDRA on real data (WikiText-103) and validate:
1. The model learns meaningful language modeling (perplexity)
2. The router learns to specialize (routing patterns)
3. HYDRA matches a compute-matched dense baseline at lower cost

**Hardware**: NVIDIA L4 (24GB VRAM) — uses AMP (fp16) for memory efficiency.

**Dataset**: WikiText-103 from HuggingFace — a standard LM benchmark.

## 0. Setup & Install Dependencies

In [2]:
!pip install datasets transformers tokenizers tqdm matplotlib einops scipy -q

import sys, os, subprocess, importlib, shutil

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

# --- Mount Google Drive for snapshots ---
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_SAVE_DIR = "/content/drive/MyDrive/HYDRA_snapshots"
    os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
    print(f"Drive mounted. Snapshots will save to: {DRIVE_SAVE_DIR}")
else:
    DRIVE_SAVE_DIR = None

# --- Clone or update project ---
if IN_COLAB:
    PROJECT_ROOT = "/content/INNOVATIONS"
    if not os.path.exists(os.path.join(PROJECT_ROOT, "hydra")):
        print("Cloning HYDRA project from GitHub...")
        subprocess.run(["git", "clone", "https://github.com/RAVINDRA8008/INNOVATIONS.git", PROJECT_ROOT], check=True)
        print(f"Project cloned to {PROJECT_ROOT}")
    else:
        print(f"Project exists, pulling latest...")
        subprocess.run(["git", "-C", PROJECT_ROOT, "reset", "--hard", "origin/main"], check=True)
        subprocess.run(["git", "-C", PROJECT_ROOT, "pull", "origin", "main"], check=True)
        print("Updated to latest version")
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if not os.path.exists(os.path.join(PROJECT_ROOT, "hydra")):
        PROJECT_ROOT = os.getcwd()
    print(f"Local project root: {PROJECT_ROOT}")

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

# Force-reload all hydra modules (clears stale imports from previous runs)
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith("hydra"):
        del sys.modules[mod_name]
print("Cleared cached hydra modules")

assert os.path.exists(os.path.join(PROJECT_ROOT, "hydra", "model", "config.py")), \
    f"ERROR: hydra package not found at {PROJECT_ROOT}/hydra/"
print(f"hydra package verified at {PROJECT_ROOT}/hydra/")

Mounted at /content/drive
Drive mounted. Snapshots will save to: /content/drive/MyDrive/HYDRA_snapshots
Cloning HYDRA project from GitHub...
Project cloned to /content/INNOVATIONS
Cleared cached hydra modules
hydra package verified at /content/INNOVATIONS/hydra/


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import time, json, math, gc

# Reduce VRAM fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from hydra.model.config import HydraConfig
from hydra.model.hydra_model import HydraModel
from hydra.training.optimizer import create_optimizer, HydraOptimizer
from hydra.training.adaptive_loss import AdaptiveComputeBudgetLoss, RoutingDiversityRegularizer
from hydra.training.curriculum import CurriculumScheduler, TemperatureCallback

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    # Enable TF32 for faster matmuls on Ampere+ GPUs (L4 is Ada Lovelace)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print("TF32 enabled for faster matmuls")

Device: cuda
GPU: NVIDIA L4
VRAM: 23.7 GB
TF32 enabled for faster matmuls


/usr/local/lib/python3.12/dist-packages/torch/backends/__init__.py:46: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  self.setter(val)


## 1. Load & Tokenize WikiText-103

In [4]:
from datasets import load_dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
VOCAB_SIZE = tokenizer.vocab_size
print(f"Tokenizer vocab size: {VOCAB_SIZE}")

dataset = load_dataset("wikitext", "wikitext-103-v1")
print(f"Train: {len(dataset['train'])} examples")
print(f"Valid: {len(dataset['validation'])} examples")
print(f"Test:  {len(dataset['test'])} examples")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

wikitext-103-v1/validation-00000-of-0000(…):   0%|          | 0.00/655k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Train: 1801350 examples
Valid: 3760 examples
Test:  4358 examples


In [5]:
SEQ_LEN = 256  # 256 instead of 512 — 4x faster attention, fits L4 easily

class WikiTextDataset(Dataset):
    """Memory-efficient dataset: tokenizes in small batches to avoid RAM spikes."""
    def __init__(self, hf_dataset, tokenizer, seq_len, max_tokens=None):
        all_tokens = []
        batch_texts = []
        batch_chars = 0
        BATCH_LIMIT = 500_000  # tokenize 500K chars at a time

        for row in hf_dataset:
            line = row["text"]
            if not line.strip():
                continue
            batch_texts.append(line)
            batch_chars += len(line)

            if batch_chars >= BATCH_LIMIT:
                toks = tokenizer.encode("\n".join(batch_texts))
                all_tokens.extend(toks)
                batch_texts = []
                batch_chars = 0
                if max_tokens and len(all_tokens) >= max_tokens:
                    all_tokens = all_tokens[:max_tokens]
                    break

        if batch_texts:
            toks = tokenizer.encode("\n".join(batch_texts))
            all_tokens.extend(toks)
            del batch_texts

        if max_tokens and len(all_tokens) > max_tokens:
            all_tokens = all_tokens[:max_tokens]

        print(f"Total tokens: {len(all_tokens):,}")
        n_chunks = len(all_tokens) // (seq_len + 1)
        all_tokens = all_tokens[:n_chunks * (seq_len + 1)]
        self.chunks = torch.tensor(all_tokens, dtype=torch.long).view(n_chunks, seq_len + 1)
        del all_tokens
        gc.collect()
        print(f"Created {n_chunks:,} chunks of length {seq_len}")

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        chunk = self.chunks[idx]
        return {"input_ids": chunk[:-1], "targets": chunk[1:]}

# Token counts sized for Colab RAM (12.7GB system)
MAX_TRAIN_TOKENS = 5_000_000    # 5M tokens — enough for 3K steps
MAX_EVAL_TOKENS = 500_000       # 500K tokens

print("\n-- Building datasets (batched tokenization) --")
train_dataset = WikiTextDataset(dataset["train"], tokenizer, SEQ_LEN, max_tokens=MAX_TRAIN_TOKENS)
eval_dataset = WikiTextDataset(dataset["validation"], tokenizer, SEQ_LEN, max_tokens=MAX_EVAL_TOKENS)
test_dataset = WikiTextDataset(dataset["test"], tokenizer, SEQ_LEN, max_tokens=MAX_EVAL_TOKENS)

# Free raw HuggingFace dataset from RAM
del dataset
gc.collect()
print("Raw dataset freed from RAM")


-- Building datasets (batched tokenization) --


Token indices sequence length is longer than the specified maximum sequence length for this model (108643 > 1024). Running this sequence through the model will result in indexing errors


Total tokens: 5,000,000
Created 19,455 chunks of length 256
Total tokens: 247,137
Created 961 chunks of length 256
Total tokens: 282,300
Created 1,098 chunks of length 256
Raw dataset freed from RAM


In [6]:
BATCH_SIZE = 32   # Doubled since SEQ_LEN halved (256 vs 512)
GRAD_ACCUM = 2
EVAL_BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=0, pin_memory=True, drop_last=True  # num_workers=0 avoids Colab multiprocessing bugs
)
eval_loader = DataLoader(
    eval_dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=True
)

print(f"Train batches: {len(train_loader):,}  (batch_size={BATCH_SIZE})")
print(f"Eval batches:  {len(eval_loader):,}  (batch_size={EVAL_BATCH_SIZE})")
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")

Train batches: 607  (batch_size=32)
Eval batches:  61  (batch_size=16)
Effective batch size: 64


## 2. Create HYDRA Model (Small Config)

Small config (~25M params) sized for L4 GPU.
Gated linear mixer, sparse mixing (every 3 layers), gradient checkpointing.

In [7]:
hydra_config = HydraConfig.small()
hydra_config.vocab_size = VOCAB_SIZE
hydra_config.max_seq_len = SEQ_LEN
hydra_config.batch_size = BATCH_SIZE
hydra_config.learning_rate = 6e-4
hydra_config.warmup_steps = 200
hydra_config.max_steps = 3000
hydra_config.dropout = 0.1
hydra_config.gradient_clip = 1.0
hydra_config.checkpoint_pathways = "non_dominant"
hydra_config.curriculum_enabled = True
hydra_config.curriculum_warmup_steps = 500

hydra_model = HydraModel(hydra_config).to(device)
print(hydra_model)
print(f"\nParameter counts:")
for k, v in hydra_model.count_parameters().items():
    if k != "per_block":
        print(f"  {k}: {v:,}")

HydraModel(
  d_model=256, n_layers=6,
  n_heads=4, d_ff=1024,
  vocab_size=50257,
  params=30,385,636 (30.4M)
)

Parameter counts:
  total: 30,385,636
  trainable: 30,385,618
  embeddings: 12,931,328


## 3. Dense Baseline (Compute-Matched Transformer)

Standard transformer with the **same parameter count** as HYDRA.
If HYDRA doesn't match or beat it, the routing overhead isn't justified.

In [ ]:
class DenseTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, max_seq_len, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.drop = nn.Dropout(dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, activation="gelu", batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight
        self.apply(self._init_weights)
        n_params = sum(p.numel() for p in self.parameters())
        print(f"Dense Transformer: {n_params:,} params ({n_params/1e6:.1f}M)")

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, std=0.02)

    def forward(self, input_ids, targets=None):
        B, L = input_ids.shape
        pos = torch.arange(L, device=input_ids.device).unsqueeze(0)
        x = self.drop(self.token_emb(input_ids) + self.pos_emb(pos))
        # Generate causal mask explicitly (required by some PyTorch versions)
        mask = nn.Transformer.generate_square_subsequent_mask(L, device=input_ids.device)
        x = self.transformer(x, mask=mask, is_causal=True)
        x = self.norm(x)
        logits = self.lm_head(x)
        result = {"logits": logits}
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
            result["loss"] = loss
        return result

dense_model = DenseTransformer(
    vocab_size=VOCAB_SIZE, d_model=256, n_layers=6,
    n_heads=4, d_ff=1024, max_seq_len=SEQ_LEN, dropout=0.1,
).to(device)

print(f"\nHYDRA params:  {hydra_model._n_params:,}")
print(f"Dense params:  {sum(p.numel() for p in dense_model.parameters()):,}")
print(f"Ratio:         {hydra_model._n_params / sum(p.numel() for p in dense_model.parameters()):.2f}x")

Dense Transformer: 17,670,400 params (17.7M)

HYDRA params:  30,385,636
Dense params:  17,670,400
Ratio:         1.72x


## 4. Training Loop

Both models trained with the same:
- Effective batch size (64)
- Learning rate schedule (warmup + cosine decay)
- 3K steps (~30 min total on L4), AMP (fp16)

In [9]:
@torch.no_grad()
def evaluate_model(model, loader, is_hydra=False, max_batches=100):
    model.eval()
    torch.cuda.empty_cache()
    total_loss = 0
    total_tokens = 0
    routing_accum = {"stream": 0, "focus": 0, "reason": 0}
    n_batches = 0
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        targets = batch["targets"].to(device)
        with torch.amp.autocast("cuda", enabled=device.type=="cuda"):
            output = model(input_ids, targets=targets)
        loss = output["loss"].item()
        total_loss += loss * input_ids.shape[0]
        total_tokens += input_ids.shape[0]
        if is_hydra:
            stats = output["routing_stats"]
            routing_accum["stream"] += stats["avg_stream_frac"]
            routing_accum["focus"] += stats["avg_focus_frac"]
            routing_accum["reason"] += stats["avg_reason_frac"]
        n_batches += 1
        if n_batches >= max_batches:
            break
        del output, input_ids, targets  # free VRAM between batches
    avg_loss = total_loss / max(total_tokens, 1)
    ppl = min(math.exp(avg_loss), 1e6)
    result = {"loss": avg_loss, "perplexity": ppl}
    if is_hydra and n_batches > 0:
        result["routing"] = {k: v / n_batches for k, v in routing_accum.items()}
    torch.cuda.empty_cache()
    model.train()
    return result

In [10]:
def save_snapshot(model, optimizer, history, step, model_name, drive_dir):
    """Save training snapshot to Google Drive."""
    if drive_dir is None:
        return
    snapshot = {
        "step": step,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "history": history,
    }
    path = os.path.join(drive_dir, f"{model_name}_step{step}.pt")
    torch.save(snapshot, path)
    # Also save a "latest" checkpoint (overwritten each time)
    latest_path = os.path.join(drive_dir, f"{model_name}_latest.pt")
    torch.save(snapshot, latest_path)
    print(f"  Snapshot saved to Drive: {path}")

def train_model(
    model, train_loader, eval_loader,
    max_steps=10000, lr=6e-4, warmup_steps=500,
    grad_accum=2, is_hydra=False, model_name="Model",
    eval_every=500, log_every=100, drive_dir=None,
):
    if is_hydra:
        hydra_opt = HydraOptimizer(model, model.config)
        optimizer = hydra_opt.optimizer
        scheduler = hydra_opt.scheduler
        adaptive_loss = AdaptiveComputeBudgetLoss()
        diversity_reg = RoutingDiversityRegularizer()
        curriculum = CurriculumScheduler(model.config)
    else:
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=0.01)
        def lr_lambda(step):
            if step < warmup_steps:
                return step / max(1, warmup_steps)
            progress = (step - warmup_steps) / max(1, max_steps - warmup_steps)
            return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * progress))
        scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    scaler = torch.amp.GradScaler("cuda") if device.type == "cuda" else None
    history = {
        "train_loss": [], "eval_loss": [], "eval_ppl": [],
        "steps": [], "eval_steps": [],
        "routing_stream": [], "routing_focus": [], "routing_reason": [],
        "lr": [], "wall_time": [],
    }
    model.train()
    data_iter = iter(train_loader)
    running_loss = 0
    t0 = time.time()
    pbar = tqdm(range(max_steps), desc=f"Training {model_name}")

    for step in pbar:
        try:
            batch = next(data_iter)
        except StopIteration:
            data_iter = iter(train_loader)
            batch = next(data_iter)

        input_ids = batch["input_ids"].to(device)
        targets = batch["targets"].to(device)

        with torch.amp.autocast("cuda", enabled=device.type=="cuda"):
            output = model(input_ids, targets=targets)
            if is_hydra:
                loss_info = adaptive_loss(output)
                div_loss = diversity_reg(output["block_info"])
                loss = (loss_info["total_loss"] + 0.01 * div_loss) / grad_accum
            else:
                loss = output["loss"] / grad_accum

        if scaler:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        if (step + 1) % grad_accum == 0:
            if scaler:
                scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            if scaler:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        if is_hydra:
            curriculum.step(model)

        raw_loss = loss.item() * grad_accum
        running_loss += raw_loss

        if (step + 1) % log_every == 0:
            avg_loss = running_loss / log_every
            history["train_loss"].append(avg_loss)
            history["steps"].append(step + 1)
            history["lr"].append(optimizer.param_groups[0]["lr"])
            history["wall_time"].append(time.time() - t0)
            if is_hydra:
                stats = output["routing_stats"]
                s, f, r = stats["avg_stream_frac"], stats["avg_focus_frac"], stats["avg_reason_frac"]
                history["routing_stream"].append(s)
                history["routing_focus"].append(f)
                history["routing_reason"].append(r)
                pbar.set_postfix_str(f"loss={avg_loss:.3f} S={s:.2f} F={f:.2f} R={r:.2f}")
            else:
                pbar.set_postfix_str(f"loss={avg_loss:.3f}")
            running_loss = 0

        if (step + 1) % eval_every == 0:
            eval_result = evaluate_model(model, eval_loader, is_hydra=is_hydra)
            history["eval_loss"].append(eval_result["loss"])
            history["eval_ppl"].append(eval_result["perplexity"])
            history["eval_steps"].append(step + 1)
            elapsed = time.time() - t0
            print(f"\n  [{model_name}] Step {step+1} | "
                  f"Eval Loss: {eval_result['loss']:.4f} | "
                  f"PPL: {eval_result['perplexity']:.2f} | "
                  f"Time: {elapsed:.0f}s")
            if is_hydra and "routing" in eval_result:
                rt = eval_result["routing"]
                print(f"           Routing: Stream={rt['stream']:.3f} "
                      f"Focus={rt['focus']:.3f} Reason={rt['reason']:.3f}")
            # Save snapshot to Drive every eval
            save_snapshot(model, optimizer, history, step + 1, model_name.replace(" ", "_"), drive_dir)

    final = evaluate_model(model, eval_loader, is_hydra=is_hydra)
    history["final_eval"] = final
    history["total_time"] = time.time() - t0
    print(f"\n{'='*60}")
    print(f"  {model_name} FINAL RESULTS")
    print(f"  Loss: {final['loss']:.4f}  |  PPL: {final['perplexity']:.2f}")
    print(f"  Wall time: {history['total_time']:.0f}s")
    print(f"{'='*60}")
    # Save final snapshot
    save_snapshot(model, optimizer, history, "final", model_name.replace(" ", "_"), drive_dir)
    return history

### 4a. Train Dense Baseline

In [11]:
MAX_STEPS = 3000

print("Training Dense Transformer baseline...")
dense_history = train_model(
    dense_model, train_loader, eval_loader,
    max_steps=MAX_STEPS, lr=6e-4, warmup_steps=200,
    grad_accum=GRAD_ACCUM, is_hydra=False,
    model_name="Dense Baseline",
    eval_every=300, log_every=50,
    drive_dir=DRIVE_SAVE_DIR,
)

Training Dense Transformer baseline...


Training Dense Baseline:   0%|          | 0/3000 [00:00<?, ?it/s]

RuntimeError: Need attn_mask if specifying the is_causal hint. You may use the Transformer module method `generate_square_subsequent_mask` to create this mask.

### 4b. Train HYDRA

In [ ]:
print("Training HYDRA...")
hydra_history = train_model(
    hydra_model, train_loader, eval_loader,
    max_steps=MAX_STEPS, lr=6e-4, warmup_steps=200,
    grad_accum=GRAD_ACCUM, is_hydra=True,
    model_name="HYDRA",
    eval_every=300, log_every=50,
    drive_dir=DRIVE_SAVE_DIR,
)

Training HYDRA...


Training HYDRA:   0%|          | 0/10000 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 5. Results Comparison

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("HYDRA vs Dense Transformer - WikiText-103", fontsize=14, fontweight="bold")

# Training Loss
ax = axes[0, 0]
ax.plot(dense_history["steps"], dense_history["train_loss"], label="Dense", alpha=0.8)
ax.plot(hydra_history["steps"], hydra_history["train_loss"], label="HYDRA", alpha=0.8)
ax.set_xlabel("Step"); ax.set_ylabel("Training Loss"); ax.set_title("Training Loss")
ax.legend(); ax.grid(True, alpha=0.3)

# Eval Perplexity
ax = axes[0, 1]
ax.plot(dense_history["eval_steps"], dense_history["eval_ppl"], "o-", label="Dense", markersize=4)
ax.plot(hydra_history["eval_steps"], hydra_history["eval_ppl"], "s-", label="HYDRA", markersize=4)
ax.set_xlabel("Step"); ax.set_ylabel("Perplexity"); ax.set_title("Eval Perplexity (lower=better)")
ax.legend(); ax.grid(True, alpha=0.3)

# Routing Distribution
ax = axes[1, 0]
if hydra_history["routing_stream"]:
    steps = hydra_history["steps"][:len(hydra_history["routing_stream"])]
    ax.plot(steps, hydra_history["routing_stream"], label="Stream (SSM)", color="#2ecc71")
    ax.plot(steps, hydra_history["routing_focus"], label="Focus (Windowed)", color="#3498db")
    ax.plot(steps, hydra_history["routing_reason"], label="Reason (Global)", color="#e74c3c")
    ax.axhline(y=0.6, color="#2ecc71", linestyle="--", alpha=0.3)
    ax.axhline(y=0.25, color="#3498db", linestyle="--", alpha=0.3)
    ax.axhline(y=0.15, color="#e74c3c", linestyle="--", alpha=0.3)
ax.set_xlabel("Step"); ax.set_ylabel("Routing Fraction")
ax.set_title("HYDRA Routing Distribution Over Training")
ax.legend(fontsize=8); ax.set_ylim(0, 1); ax.grid(True, alpha=0.3)

# Final PPL Bar Chart
ax = axes[1, 1]
models_list = ["Dense", "HYDRA"]
times = [dense_history["total_time"], hydra_history["total_time"]]
final_ppl = [dense_history["final_eval"]["perplexity"], hydra_history["final_eval"]["perplexity"]]
colors = ["#3498db", "#e74c3c"]
bars = ax.bar(models_list, final_ppl, color=colors, alpha=0.8)
ax.set_ylabel("Final Eval Perplexity"); ax.set_title("Final Perplexity Comparison")
for bar, ppl, t in zip(bars, final_ppl, times):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"PPL={ppl:.1f}\n({t:.0f}s)", ha="center", va="bottom", fontsize=9)
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
os.makedirs("outputs", exist_ok=True)
plt.savefig("outputs/hydra_vs_dense.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\n{'='*60}")
print("  SUMMARY")
print(f"{'='*60}")
print(f"  Dense:  PPL={final_ppl[0]:.2f}  Time={times[0]:.0f}s")
print(f"  HYDRA:  PPL={final_ppl[1]:.2f}  Time={times[1]:.0f}s")
ppl_delta = final_ppl[1] - final_ppl[0]
print(f"  PPL difference: {ppl_delta:+.2f} ({'worse' if ppl_delta > 0 else 'better'})")
print(f"  Speed ratio:    {times[1]/times[0]:.2f}x ({'slower' if times[1] > times[0] else 'faster'})")
print(f"{'='*60}")

## 6. Routing Pattern Analysis

**Key question**: Does the router learn to send different token types
to different pathways? Uniform routing = routing is useless.

In [ ]:
sample_batch = next(iter(eval_loader))
sample_ids = sample_batch["input_ids"][:4].to(device)

hydra_model.eval()
with torch.no_grad():
    routing_info = hydra_model.get_routing_map(sample_ids)

routing_map = routing_info["routing_map"]
decisions = routing_info["decisions"]
complexity = routing_info["complexity_map"]

print(f"Routing map shape: {routing_map.shape}")
print(f"Decisions shape:   {decisions.shape}")

In [ ]:
from matplotlib.patches import Patch

sample_idx = 0
dec = decisions[:, sample_idx, :].cpu().numpy()
n_layers_vis, seq_len_vis = dec.shape

SHOW_TOKENS = min(80, seq_len_vis)
token_ids = sample_ids[sample_idx, :SHOW_TOKENS].cpu().tolist()
token_strs = [tokenizer.decode([t]).replace("\n", "\\n") for t in token_ids]

fig, ax = plt.subplots(1, 1, figsize=(20, 4))
cmap = plt.cm.colors.ListedColormap(["#2ecc71", "#3498db", "#e74c3c"])
ax.imshow(dec[:, :SHOW_TOKENS], cmap=cmap, aspect="auto", vmin=0, vmax=2)
ax.set_xlabel("Token"); ax.set_ylabel("Layer")
ax.set_title("HYDRA Routing Decisions - Per Token Per Layer")
ax.set_yticks(range(n_layers_vis))
ax.set_yticklabels([f"Layer {i}" for i in range(n_layers_vis)])
step_vis = max(1, SHOW_TOKENS // 40)
ax.set_xticks(range(0, SHOW_TOKENS, step_vis))
ax.set_xticklabels(token_strs[::step_vis], rotation=60, ha="right", fontsize=7)
legend_elements = [
    Patch(facecolor="#2ecc71", label="Stream (SSM)"),
    Patch(facecolor="#3498db", label="Focus (Windowed)"),
    Patch(facecolor="#e74c3c", label="Reason (Global)"),
]
ax.legend(handles=legend_elements, loc="upper right", fontsize=8)
plt.tight_layout()
plt.savefig("outputs/routing_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
all_tokens = []
all_decisions_list = []

hydra_model.eval()
for i, batch in enumerate(eval_loader):
    if i >= 20:
        break
    ids = batch["input_ids"].to(device)
    with torch.no_grad():
        rmap = hydra_model.get_routing_map(ids)
    last_layer_dec = rmap["decisions"][-1]
    all_tokens.append(ids.cpu())
    all_decisions_list.append(last_layer_dec.cpu())

all_tokens_t = torch.cat(all_tokens, dim=0).flatten()
all_decisions_t = torch.cat(all_decisions_list, dim=0).flatten()

pathway_names = ["Stream (SSM)", "Focus (Windowed)", "Reason (Global)"]

print("\nOverall routing distribution (last layer):")
for i, name in enumerate(pathway_names):
    frac = (all_decisions_t == i).float().mean().item()
    print(f"  {name}: {frac:.1%}")

simple_tokens = tokenizer.encode(" the a an of to in is was it for on with")
complex_tokens = tokenizer.encode(" therefore however consequently although nevertheless")

print("\nRouting by token type:")
for category, token_set in [
    ("Simple (articles/prep)", simple_tokens),
    ("Complex (connectives)", complex_tokens),
]:
    mask = torch.zeros(len(all_tokens_t), dtype=torch.bool)
    for t in token_set:
        mask |= (all_tokens_t == t)
    if mask.sum() > 0:
        decisions_subset = all_decisions_t[mask]
        print(f"\n  {category} ({mask.sum()} tokens):")
        for i, pname in enumerate(pathway_names):
            frac = (decisions_subset == i).float().mean().item()
            bar = chr(9608) * int(frac * 40)
            print(f"    {pname:25s} {frac:5.1%} {bar}")

## 7. Test Set Evaluation

In [ ]:
print("Evaluating on test set...")
dense_test = evaluate_model(dense_model, test_loader, is_hydra=False, max_batches=200)
hydra_test = evaluate_model(hydra_model, test_loader, is_hydra=True, max_batches=200)

print(f"\n{'='*60}")
print("  TEST SET RESULTS (WikiText-103)")
print(f"{'='*60}")
print("  Dense Baseline:")
print(f"    Loss:       {dense_test['loss']:.4f}")
print(f"    Perplexity: {dense_test['perplexity']:.2f}")
print()
print("  HYDRA:")
print(f"    Loss:       {hydra_test['loss']:.4f}")
print(f"    Perplexity: {hydra_test['perplexity']:.2f}")
if "routing" in hydra_test:
    rt = hydra_test["routing"]
    print(f"    Routing:    Stream={rt['stream']:.3f}  Focus={rt['focus']:.3f}  Reason={rt['reason']:.3f}")
print()
ppl_diff = hydra_test["perplexity"] - dense_test["perplexity"]
print(f"  Delta PPL:    {ppl_diff:+.2f} ({'HYDRA wins' if ppl_diff < 0 else 'Dense wins'})")
print(f"{'='*60}")

## 8. Save Results

In [ ]:
os.makedirs("outputs", exist_ok=True)

results = {
    "config": {
        "seq_len": SEQ_LEN,
        "batch_size": BATCH_SIZE,
        "grad_accum": GRAD_ACCUM,
        "max_steps": MAX_STEPS,
        "hydra_params": hydra_model._n_params,
        "dense_params": sum(p.numel() for p in dense_model.parameters()),
    },
    "dense": {
        "final_eval_loss": dense_test["loss"],
        "final_eval_ppl": dense_test["perplexity"],
        "train_time_s": dense_history["total_time"],
    },
    "hydra": {
        "final_eval_loss": hydra_test["loss"],
        "final_eval_ppl": hydra_test["perplexity"],
        "train_time_s": hydra_history["total_time"],
        "final_routing": hydra_test.get("routing", {}),
    },
    "ppl_difference": hydra_test["perplexity"] - dense_test["perplexity"],
}

with open("outputs/training_results.json", "w") as f:
    json.dump(results, f, indent=2)

torch.save(hydra_model.state_dict(), "outputs/hydra_wikitext.pt")
torch.save(dense_model.state_dict(), "outputs/dense_baseline_wikitext.pt")

print("Saved locally:")
print("  outputs/training_results.json")
print("  outputs/hydra_vs_dense.png")
print("  outputs/routing_heatmap.png")
print("  outputs/hydra_wikitext.pt")
print("  outputs/dense_baseline_wikitext.pt")

# Copy all outputs to Google Drive
if DRIVE_SAVE_DIR:
    import shutil
    for fname in os.listdir("outputs"):
        src = os.path.join("outputs", fname)
        dst = os.path.join(DRIVE_SAVE_DIR, fname)
        shutil.copy2(src, dst)
        print(f"  -> Drive: {dst}")
    # Also save the processed datasets to Drive
    dataset_dir = os.path.join(DRIVE_SAVE_DIR, "processed_datasets")
    os.makedirs(dataset_dir, exist_ok=True)
    torch.save(train_dataset.chunks, os.path.join(dataset_dir, "train_chunks.pt"))
    torch.save(eval_dataset.chunks, os.path.join(dataset_dir, "eval_chunks.pt"))
    torch.save(test_dataset.chunks, os.path.join(dataset_dir, "test_chunks.pt"))
    print(f"\nProcessed datasets saved to Drive: {dataset_dir}")
    print(f"  train_chunks.pt: {train_dataset.chunks.shape}")
    print(f"  eval_chunks.pt:  {eval_dataset.chunks.shape}")
    print(f"  test_chunks.pt:  {test_dataset.chunks.shape}")
    print(f"\nAll results backed up to: {DRIVE_SAVE_DIR}")